# 03 — Analysis & Figures

Produces the three core figures from the aggregated monthly data (`data/processed/monthly_frames.csv`).

- **Figure 1** — Monthly coverage volume (RQ1), with DOC API cross-validation overlay
- **Figure 2** — Frame distribution over time (RQ2)
- **Figure 3** — Event study around the EU AI Act (RQ3)
- **Figure 4** (optional) — Regional comparison — requires the full raw corpus

Figures are saved to `../paper/figures/` as PDF.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path('..')))

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

from src.analysis import (
    load_agg,
    monthly_volume_agg, frame_shares_agg, event_study_agg,
)
from src.doc_api import fetch_timeline_vol, build_doc_query
from src.dictionaries import MILESTONES

FIGURES = Path('../paper/figures')
FIGURES.mkdir(parents=True, exist_ok=True)

sns.set_theme(style='whitegrid', palette='colorblind')
MILESTONE_DATES = {m['name']: pd.Timestamp(m['date']) for m in MILESTONES}

In [ ]:
agg_df = load_agg('../data/processed/monthly_frames.csv')
print(f'{len(agg_df)} months, {agg_df["total_articles"].sum():,} total articles')

## Figure 1 — Monthly coverage volume (RQ1)

In [ ]:
vol = monthly_volume_agg(agg_df)
vol_dates = vol.index.to_timestamp()

# Optional: DOC API cross-validation series
try:
    vol_api = fetch_timeline_vol(build_doc_query(), start='20221101000000', end='20260601000000')
    # Normalise to same scale for overlay
    api_dates = vol_api['month'].dt.to_timestamp()
    api_scaled = vol_api['volume'] / vol_api['volume'].max() * vol.max()
    has_api = True
except Exception as e:
    print(f'DOC API unavailable ({e}), skipping overlay')
    has_api = False

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(vol_dates, vol.values, linewidth=1.8, label='BigQuery corpus', color='steelblue')
if has_api:
    ax.plot(api_dates, api_scaled, linewidth=1, linestyle='--',
            alpha=0.6, label='DOC API (normalised)', color='darkorange')

for name, ts in MILESTONE_DATES.items():
    ax.axvline(ts, color='grey', linestyle=':', linewidth=0.9)
    ax.text(ts, ax.get_ylim()[1] * 0.93, name.replace('_', ' '),
            rotation=90, fontsize=7, ha='right', va='top', color='grey')

ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
plt.xticks(rotation=45, ha='right')
ax.set_xlabel('Month')
ax.set_ylabel('Number of articles')
ax.set_title('Monthly volume of generative AI governance news in GDELT GKG')
if has_api:
    ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig(FIGURES / 'fig1_monthly_volume.pdf', bbox_inches='tight')
plt.show()
print('Saved fig1_monthly_volume.pdf')

## Figure 2 — Frame distribution over time (RQ2)

In [ ]:
shares = frame_shares_agg(agg_df)
shares_dates = shares.index.to_timestamp()

fig, ax = plt.subplots(figsize=(12, 5))
ax.stackplot(
    shares_dates,
    [shares[col] for col in shares.columns],
    labels=[c.replace('_', ' ').title() for c in shares.columns],
    alpha=0.85,
)
for name, ts in MILESTONE_DATES.items():
    ax.axvline(ts, color='white', linestyle=':', linewidth=0.9, alpha=0.8)

ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
plt.xticks(rotation=45, ha='right')
ax.set_xlabel('Month')
ax.set_ylabel('Frame share')
ax.set_ylim(0, 1)
ax.set_title('Governance frame distribution in generative AI news over time')
ax.legend(loc='upper left', fontsize=8, ncol=2)
plt.tight_layout()
plt.savefig(FIGURES / 'fig2_frame_shares.pdf', bbox_inches='tight')
plt.show()
print('Saved fig2_frame_shares.pdf')

## Figure 3 — Event study: EU AI Act (RQ3)

In [ ]:
es = event_study_agg(agg_df, 'eu_ai_act', window=3)
print(f"Milestone: {es['milestone']['description']}")
print(es['volume'])

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Left panel: volume
vol_es = es['volume']
colors = ['tomato' if i == 0 else 'steelblue' for i in vol_es.index]
axes[0].bar(vol_es.index, vol_es.values, color=colors)
axes[0].axvline(0, color='red', linestyle='--', linewidth=1.2, label='Event')
axes[0].set_xlabel('Months relative to EU AI Act adoption')
axes[0].set_ylabel('Article count')
axes[0].set_title('Coverage volume')
axes[0].set_xticks(range(-3, 4))
axes[0].legend(fontsize=8)

# Right panel: frame shares
shares_es = es['shares']
shares_es.plot(kind='bar', stacked=True, ax=axes[1], legend=True, colormap='tab10')
axes[1].axvline(3, color='red', linestyle='--', linewidth=1.2)  # rel_month 0 is at position 3
axes[1].set_xticklabels([str(i) for i in range(-3, 4)], rotation=0)
axes[1].set_xlabel('Months relative to EU AI Act adoption')
axes[1].set_ylabel('Frame share')
axes[1].set_title('Frame distribution')
axes[1].legend(fontsize=7, loc='upper left', bbox_to_anchor=(1, 1))

fig.suptitle('Event study: EU AI Act (March 2024)', fontsize=11, y=1.02)
plt.tight_layout()
plt.savefig(FIGURES / 'fig3_event_study_eu_ai_act.pdf', bbox_inches='tight')
plt.show()
print('Saved fig3_event_study_eu_ai_act.pdf')

In [ ]:
# Optional: run event study for all milestones
from src.analysis import event_study_agg

for m in MILESTONES:
    try:
        result = event_study_agg(agg_df, m['name'])
        print(f"{m['name']}: peak month rel={result['volume'].idxmax()}, count={result['volume'].max()}")
    except ValueError as e:
        print(f"{m['name']}: {e}")

## Figure 4 (optional) — Regional comparison

Requires the full raw corpus (`data/interim/gdelt_preprocessed.parquet`) from notebook 02 Part B.  
Skip if you only ran the aggregated path.

In [ ]:
RAW_PROCESSED = Path('../data/interim/gdelt_preprocessed.parquet')
if not RAW_PROCESSED.exists():
    print('Preprocessed raw corpus not found — skipping regional comparison.')
else:
    from src.analysis import region_comparison

    df_raw = pd.read_parquet(RAW_PROCESSED)
    reg = region_comparison(df_raw)

    fig, ax = plt.subplots(figsize=(8, 4))
    reg.plot(kind='bar', stacked=True, ax=ax, colormap='tab10')
    ax.set_xlabel('Region')
    ax.set_ylabel('Frame share')
    ax.set_title('Frame distribution by region (US / EU / UK)')
    ax.legend(fontsize=8, loc='upper right')
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.savefig(FIGURES / 'fig4_regional_comparison.pdf', bbox_inches='tight')
    plt.show()
    print('Saved fig4_regional_comparison.pdf')